# Benchmark Comparison

Visual comparison of model + harness benchmark results.

**Comparability rules (ADR-0002):** only compare runs within the same `capability_mode` and `telemetry_trust`. Cross-cohort comparisons are invalid.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

RESULTS = Path('results')
CSVS = ['metrics_springboot.csv', 'metrics_angular.csv', 'metrics_react.csv', 'metrics_data.csv']

dfs = []
for csv in CSVS:
    p = RESULTS / csv
    if p.exists():
        dfs.append(pd.read_csv(p))

df = pd.concat(dfs, ignore_index=True)

# Normalize booleans
df['build_ok'] = df['build_ok'].astype(str).str.strip()
df['test_ok'] = df['test_ok'].astype(str).str.strip()
df['passed'] = (df['build_ok'] == 'True') & (df['test_ok'] == 'True')
df['failed'] = (df['build_ok'] == 'False') | (df['test_ok'] == 'False')
df['errored'] = (df['build_ok'] == 'ERROR') | (df['test_ok'] == 'ERROR')

# Numeric coercion
df['in_tok'] = pd.to_numeric(df['in_tok'], errors='coerce')
df['out_tok'] = pd.to_numeric(df['out_tok'], errors='coerce')
df['cost_usd'] = pd.to_numeric(df['cost_usd'], errors='coerce')
df['latency_s'] = pd.to_numeric(df['latency_s'], errors='coerce')
df['model_calls'] = pd.to_numeric(df['model_calls'], errors='coerce')

# Short model names for charts
df['model_short'] = df['model'].str.split('/').str[-1]

print(f'{len(df)} rows loaded')
print(f"Harnesses: {df['harness'].unique().tolist()}")
print(f"Capability modes: {df['capability_mode'].unique().tolist()}")
print(f"Telemetry trust: {df['telemetry_trust'].unique().tolist()}")
print(f"Models: {sorted(df['model_short'].unique().tolist())}")
print(f"Tasks: {sorted(df['task'].unique().tolist())}")

## 1. Pass rate by model × task

Heatmap of test pass rate (3 runs per cell). White = all pass, dark = all fail. Shows which tasks discriminate between models.

In [ ]:
cohort = df[df['capability_mode'] == 'single_shot']

pivot = cohort.pivot_table(
    index='model_short', columns='task', values='passed',
    aggfunc='mean', sort=False
)

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(pivot, annot=True, fmt='.0%', cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Pass rate'})
ax.set_title('Pass rate by model × task (single-shot cohort)')
ax.set_xlabel('')
ax.set_ylabel('')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 2. Pass rate summary by model

Grouped bar: pass / fail / error across all tasks. Sorted by total pass rate.

In [ ]:
summary = cohort.groupby('model_short').agg(
    pass_rate=('passed', 'mean'),
    fail_rate=('failed', 'mean'),
    error_rate=('errored', 'mean'),
    runs=('passed', 'count'),
).sort_values('pass_rate', ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
x = range(len(summary))
ax.bar(x, summary['pass_rate'], label='Pass', color='#2ecc71')
ax.bar(x, summary['fail_rate'], bottom=summary['pass_rate'], label='Fail', color='#e74c3c')
ax.bar(x, summary['error_rate'],
       bottom=summary['pass_rate'] + summary['fail_rate'],
       label='Error', color='#95a5a6')
ax.set_xticks(list(x))
ax.set_xticklabels(summary.index, rotation=35, ha='right')
ax.set_ylabel('Rate')
ax.set_title('Outcome by model (single-shot cohort)')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
for i, (pr, n) in enumerate(zip(summary['pass_rate'], summary['runs'])):
    ax.text(i, pr / 2, f'{pr:.0%}\n(n={n})', ha='center', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 3. Cost per run by model

Box plot of `cost_usd` per run. Only within `telemetry_trust=exact` cohort (raw API). Parsed/blank cohorts omitted — different cost bases are not comparable (ADR-0002).

In [ ]:
cost_df = df[(df['telemetry_trust'] == 'exact') & df['cost_usd'].notna() & (df['cost_usd'] > 0)]

if len(cost_df) == 0:
    print('No exact-trust cost data available')
else:
    order = cost_df.groupby('model_short')['cost_usd'].median().sort_values().index
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=cost_df, x='model_short', y='cost_usd', order=order, ax=ax)
    ax.set_title('Cost per run (telemetry_trust=exact, single-shot)')
    ax.set_xlabel('')
    ax.set_ylabel('Cost (USD)')
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('$%.4f'))
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

## 4. Latency by model

Box plot of `latency_s` per run, within the single-shot cohort.

In [ ]:
lat_df = cohort[cohort['latency_s'].notna() & (cohort['latency_s'] > 0)]

order = lat_df.groupby('model_short')['latency_s'].median().sort_values().index
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=lat_df, x='model_short', y='latency_s', order=order, ax=ax)
ax.set_title('Latency per run (single-shot cohort)')
ax.set_xlabel('')
ax.set_ylabel('Latency (seconds)')
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()

## 5. Token usage: input vs output

Scatter of `in_tok` vs `out_tok`, colored by model. Only `telemetry_trust=exact` (raw API reports both reliably).

In [ ]:
tok_df = df[(df['telemetry_trust'] == 'exact')
            & df['in_tok'].notna() & df['out_tok'].notna()
            & (df['in_tok'] > 0) & (df['out_tok'] > 0)]

if len(tok_df) == 0:
    print('No exact-trust token data available')
else:
    fig, ax = plt.subplots(figsize=(10, 7))
    sns.scatterplot(data=tok_df, x='in_tok', y='out_tok',
                    hue='model_short', style='task', alpha=0.7, ax=ax)
    ax.set_title('Token usage: input vs output (telemetry_trust=exact)')
    ax.set_xlabel('Input tokens')
    ax.set_ylabel('Output tokens')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    plt.tight_layout()
    plt.show()

## 6. Cost vs latency trade-off

Each point is one model's median cost and latency. Diagonal = cheap-and-fast is better (lower-left). Only `telemetry_trust=exact`.

In [ ]:
trade = cost_df.groupby('model_short').agg(
    median_cost=('cost_usd', 'median'),
    median_latency=('latency_s', 'median'),
    pass_rate=('passed', 'mean'),
).reset_index()

if len(trade) == 0:
    print('No data for trade-off chart')
else:
    fig, ax = plt.subplots(figsize=(9, 7))
    scatter = ax.scatter(trade['median_latency'], trade['median_cost'],
                        c=trade['pass_rate'], cmap='RdYlGn',
                        s=120, edgecolors='black', linewidth=0.5,
                        vmin=0, vmax=1)
    for _, row in trade.iterrows():
        ax.annotate(row['model_short'], (row['median_latency'], row['median_cost']),
                    fontsize=8, ha='left', va='bottom',
                    xytext=(4, 4), textcoords='offset points')
    ax.set_title('Cost vs latency trade-off (median, telemetry_trust=exact)')
    ax.set_xlabel('Median latency (s)')
    ax.set_ylabel('Median cost (USD)')
    ax.yaxis.set_major_formatter(ticker.FormatStrFormatter('$%.4f'))
    fig.colorbar(scatter, label='Pass rate')
    plt.tight_layout()
    plt.show()

## 7. Pass rate by task (discrimination map)

Which tasks actually separate models. Tasks with all-pass give no signal; tasks with variance are the discriminators.

In [ ]:
task_stats = cohort.groupby('task').agg(
    pass_rate=('passed', 'mean'),
    n_runs=('passed', 'count'),
    n_models=('model_short', 'nunique'),
).sort_values('pass_rate')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if r < 0.5 else '#f39c12' if r < 1.0 else '#2ecc71'
          for r in task_stats['pass_rate']]
ax.barh(task_stats.index, task_stats['pass_rate'], color=colors)
ax.set_xlabel('Pass rate')
ax.set_title('Task discrimination map (single-shot cohort)')
ax.set_xlim(0, 1.05)
for i, (rate, n, nm) in enumerate(zip(task_stats['pass_rate'],
                                      task_stats['n_runs'],
                                      task_stats['n_models'])):
    ax.text(rate + 0.01, i, f'{rate:.0%} ({nm} models, {n} runs)',
            va='center', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Harness comparison (when agent data exists)

Grouped bar of pass rate by model × harness. Only renders if agent-iterated runs are present. **Never compare across `capability_mode` lines** — raw_api and agent bars are in separate panels.

In [ ]:
for mode in df['capability_mode'].dropna().unique():
    sub = df[df['capability_mode'] == mode]
    if sub['harness'].nunique() < 2:
        continue
    pivot = sub.pivot_table(
        index='model_short', columns='harness',
        values='passed', aggfunc='mean', sort=False
    )
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot.bar(ax=ax, width=0.8)
    ax.set_title(f'Pass rate by model × harness ({mode})')
    ax.set_xlabel('')
    ax.set_ylabel('Pass rate')
    ax.set_ylim(0, 1.05)
    ax.legend(title='Harness', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=35, ha='right')
    plt.tight_layout()
    plt.show()

if df['capability_mode'].nunique() < 2 or df['harness'].nunique() < 2:
    print('Note: only one capability mode or harness present in current data.')
    print('Run agent harnesses to populate this chart.')

## 9. Summary table

Per-model summary within the single-shot, exact-telemetry cohort.

In [ ]:
final = cohort[cohort['telemetry_trust'] == 'exact'].groupby('model_short').agg(
    runs=('passed', 'count'),
    pass_rate=('passed', 'mean'),
    median_cost=('cost_usd', 'median'),
    median_latency=('latency_s', 'median'),
    median_in_tok=('in_tok', 'median'),
    median_out_tok=('out_tok', 'median'),
).sort_values('pass_rate', ascending=False)

final.style.format({
    'pass_rate': '{:.0%}',
    'median_cost': '${:.4f}',
    'median_latency': '{:.1f}s',
    'median_in_tok': '{:.0f}',
    'median_out_tok': '{:.0f}',
})